# SageMaker Pipeline: sg-finetune

This notebook creates and executes a SageMaker Pipeline that replicates the sg-finetune workflow:

1. **Generate Dataset**: Create synthetic Catalan greeting/response pairs
2. **Train Model**: Fine-tune DistilGPT2 using HuggingFace Trainer
3. **Register Model**: Register trained model in SageMaker Model Registry

## Architecture

```
┌─────────────────┐     ┌─────────────────┐     ┌─────────────────┐
│  Generate Data  │ ──▶ │   Train Model   │ ──▶ │ Register Model  │
│  (Processing)   │     │   (Training)    │     │   (Registry)    │
└─────────────────┘     └─────────────────┘     └─────────────────┘
     ml.t3.medium         ml.g4dn.xlarge        Model Package Group
```

## Setup

In [ ]:
# Install required packages (if not already installed)
!pip install -q 'sagemaker>=2.0.0,<3.0.0'

In [ ]:
import json
import os
from datetime import datetime

import boto3
import sagemaker
from sagemaker.huggingface import HuggingFace
from sagemaker.processing import ProcessingOutput, ScriptProcessor
from sagemaker.workflow.parameters import ParameterInteger, ParameterString
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.pipeline_context import PipelineSession
from sagemaker.workflow.steps import ProcessingStep, TrainingStep
from sagemaker.workflow.step_collections import RegisterModel

print(f"SageMaker SDK version: {sagemaker.__version__}")

In [ ]:
# Configuration
REGION = "eu-west-1"
PIPELINE_NAME = "sg-finetune-pipeline"
BASE_JOB_PREFIX = "sg-finetune"

# Initialize sessions
boto_session = boto3.Session(region_name=REGION)
sagemaker_session = PipelineSession(boto_session=boto_session)

# Get default bucket and role
default_bucket = sagemaker_session.default_bucket()

# Get execution role (from SageMaker Studio or custom role)
try:
    role = sagemaker.get_execution_role()
except ValueError:
    # Fallback to sg-finetune role
    sts = boto3.client("sts", region_name=REGION)
    account_id = sts.get_caller_identity()["Account"]
    role = f"arn:aws:iam::{account_id}:role/sg-finetune-sagemaker-role"

print(f"Region: {REGION}")
print(f"Bucket: {default_bucket}")
print(f"Role: {role}")

## Pipeline Parameters

These parameters can be overridden at execution time.

In [ ]:
# Pipeline parameters (can be overridden at execution time)
num_examples = ParameterInteger(name="NumExamples", default_value=500)
train_ratio = ParameterString(name="TrainRatio", default_value="0.9")
model_id = ParameterString(name="ModelId", default_value="distilgpt2")
instance_type = ParameterString(name="InstanceType", default_value="ml.g4dn.xlarge")
epochs = ParameterInteger(name="Epochs", default_value=5)
batch_size = ParameterInteger(name="BatchSize", default_value=8)
learning_rate = ParameterString(name="LearningRate", default_value="2e-5")
max_length = ParameterInteger(name="MaxLength", default_value=128)
approval_status = ParameterString(name="ModelApprovalStatus", default_value="PendingManualApproval")

## Step 1: Generate Dataset (Processing Step)

This step generates synthetic Catalan greeting/response pairs for training.

In [ ]:
# Create preprocessing script
preprocess_script = """#!/usr/bin/env python3
import argparse
import json
import os
import random

GREETINGS = [
    "bon dia", "Bon dia", "Bon dia!", "bon dia!", "BON DIA", "Bon Dia",
    "hola, bon dia", "Hola, bon dia!", "ei, bon dia", "Ei, bon dia!",
    "hey, bon dia", "bones, bon dia", "que tal, bon dia", "bon dia a tots",
    "bon dia a tothom", "molt bon dia", "bon dia, com estàs?", "bon dia, què tal?",
]

RESPONSES = [
    "Serà per tu!", "serà per tu!", "Serà per tu",
    "I tant, serà per tu!", "Segur que serà per tu!",
]

GREETING_SUFFIXES = ["", " ", "  ", "\\n"]

def generate_training_example(greeting, response):
    text = f"### Input:\\n{greeting}\\n\\n### Response:\\n{response}"
    return {"text": text}

def generate_dataset(num_examples, train_ratio, output_dir, seed):
    random.seed(seed)
    os.makedirs(output_dir, exist_ok=True)
    
    examples = []
    while len(examples) < num_examples:
        greeting = random.choice(GREETINGS)
        response = random.choice(RESPONSES)
        if random.random() < 0.1:
            greeting = greeting.lower()
        if random.random() < 0.1:
            greeting = greeting.upper()
        if random.random() < 0.2:
            greeting = greeting + random.choice(GREETING_SUFFIXES)
        examples.append(generate_training_example(greeting, response))
    
    random.shuffle(examples)
    split_idx = int(len(examples) * train_ratio)
    train_examples = examples[:split_idx]
    val_examples = examples[split_idx:]
    
    with open(os.path.join(output_dir, "train.jsonl"), "w", encoding="utf-8") as f:
        for ex in train_examples:
            f.write(json.dumps(ex, ensure_ascii=False) + "\\n")
    
    with open(os.path.join(output_dir, "validation.jsonl"), "w", encoding="utf-8") as f:
        for ex in val_examples:
            f.write(json.dumps(ex, ensure_ascii=False) + "\\n")
    
    print(f"Generated {len(train_examples)} training, {len(val_examples)} validation examples")

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--num-examples", type=int, default=500)
    parser.add_argument("--train-ratio", type=float, default=0.9)
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()
    generate_dataset(args.num_examples, args.train_ratio, "/opt/ml/processing/output", args.seed)
"""

# Save preprocessing script locally
os.makedirs("scripts", exist_ok=True)
with open("scripts/preprocess.py", "w") as f:
    f.write(preprocess_script)

print("Preprocessing script saved to scripts/preprocess.py")

In [ ]:
# Create processor for data generation
sklearn_processor = ScriptProcessor(
    image_uri=sagemaker.image_uris.retrieve(
        framework="sklearn",
        region=REGION,
        version="1.2-1",
        instance_type="ml.t3.medium",
    ),
    instance_type="ml.t3.medium",
    instance_count=1,
    base_job_name=f"{BASE_JOB_PREFIX}-preprocess",
    role=role,
    sagemaker_session=sagemaker_session,
)

step_preprocess = ProcessingStep(
    name="GenerateDataset",
    processor=sklearn_processor,
    outputs=[
        ProcessingOutput(
            output_name="training_data",
            source="/opt/ml/processing/output",
            destination=f"s3://{default_bucket}/{BASE_JOB_PREFIX}/pipeline/data",
        )
    ],
    code="scripts/preprocess.py",
    job_arguments=[
        "--num-examples", num_examples.to_string(),
        "--train-ratio", train_ratio,
        "--seed", "42",
    ],
)

print("Step 1: GenerateDataset configured")

## Step 2: Train Model (Training Step)

This step fine-tunes DistilGPT2 on the generated dataset.

In [ ]:
# Create training script
train_script = '''#!/usr/bin/env python3
"""SageMaker training script for fine-tuning DistilGPT2."""

import json
import os
from pathlib import Path

import torch
from datasets import Dataset, DatasetDict
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)


def load_dataset_from_jsonl(data_dir):
    data_path = Path(data_dir)
    
    def load_jsonl(file_path):
        data = []
        with open(file_path, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    data.append(json.loads(line))
        return data
    
    train_data = load_jsonl(data_path / "train.jsonl")
    val_data = load_jsonl(data_path / "validation.jsonl")
    
    train_dataset = Dataset.from_dict({"text": [item["text"] for item in train_data]})
    val_dataset = Dataset.from_dict({"text": [item["text"] for item in val_data]})
    
    return DatasetDict({"train": train_dataset, "validation": val_dataset})


def tokenize_function(examples, tokenizer, max_length=128):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
    )


def main():
    model_id = os.environ.get("SM_HP_MODEL_ID", "distilgpt2")
    learning_rate = float(os.environ.get("SM_HP_LEARNING_RATE", "2e-5"))
    batch_size = int(os.environ.get("SM_HP_BATCH_SIZE", "8"))
    epochs = int(os.environ.get("SM_HP_EPOCHS", "5"))
    max_length = int(os.environ.get("SM_HP_MAX_LENGTH", "128"))
    warmup_steps = int(os.environ.get("SM_HP_WARMUP_STEPS", "50"))
    weight_decay = float(os.environ.get("SM_HP_WEIGHT_DECAY", "0.01"))
    
    data_dir = os.environ.get("SM_CHANNEL_TRAINING", "/opt/ml/input/data/training")
    model_dir = os.environ.get("SM_MODEL_DIR", "/opt/ml/model")
    output_dir = os.environ.get("SM_OUTPUT_DATA_DIR", "/opt/ml/output/data")
    
    print(f"Model: {model_id}, Epochs: {epochs}, Batch: {batch_size}")
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device: {device}")
    
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id)
    
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        model.config.pad_token_id = model.config.eos_token_id
    
    dataset = load_dataset_from_jsonl(data_dir)
    print(f"Train: {len(dataset[\"train\"])}, Val: {len(dataset[\"validation\"])}")
    
    tokenized_dataset = dataset.map(
        lambda x: tokenize_function(x, tokenizer, max_length),
        batched=True,
        remove_columns=["text"],
    )
    
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
    
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        learning_rate=learning_rate,
        warmup_steps=warmup_steps,
        weight_decay=weight_decay,
        logging_steps=10,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        fp16=torch.cuda.is_available(),
        report_to="none",
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["validation"],
        data_collator=data_collator,
    )
    
    trainer.train()
    eval_results = trainer.evaluate()
    print(f"Eval results: {eval_results}")
    
    trainer.save_model(model_dir)
    tokenizer.save_pretrained(model_dir)
    
    with open(Path(model_dir) / "training_metrics.json", "w") as f:
        json.dump(eval_results, f, indent=2)
    
    print(f"Model saved to {model_dir}")


if __name__ == "__main__":
    main()
'''

# Save training script
os.makedirs("src", exist_ok=True)
with open("src/train.py", "w") as f:
    f.write(train_script)

# Create requirements.txt
requirements = """transformers==4.36.0
datasets>=2.14.0
accelerate>=0.21.0
evaluate>=0.4.0
"""

with open("src/requirements.txt", "w") as f:
    f.write(requirements)

print("Training script saved to src/train.py")

In [ ]:
# Create HuggingFace estimator
huggingface_estimator = HuggingFace(
    entry_point="train.py",
    source_dir="src",
    instance_type=instance_type,
    instance_count=1,
    role=role,
    transformers_version="4.36.0",
    pytorch_version="2.1.0",
    py_version="py310",
    hyperparameters={
        "model_id": model_id,
        "learning_rate": learning_rate,
        "batch_size": batch_size,
        "epochs": epochs,
        "max_length": max_length,
        "warmup_steps": 50,
        "weight_decay": 0.01,
    },
    output_path=f"s3://{default_bucket}/{BASE_JOB_PREFIX}/pipeline/models",
    base_job_name=f"{BASE_JOB_PREFIX}-train",
    max_run=3600,
    sagemaker_session=sagemaker_session,
)

step_train = TrainingStep(
    name="TrainModel",
    estimator=huggingface_estimator,
    inputs={
        "training": sagemaker.inputs.TrainingInput(
            s3_data=step_preprocess.properties.ProcessingOutputConfig.Outputs[
                "training_data"
            ].S3Output.S3Uri,
            content_type="application/jsonlines",
        )
    },
)

print("Step 2: TrainModel configured")

## Step 3: Register Model (Model Registry)

This step registers the trained model in SageMaker Model Registry.

In [ ]:
# Get inference container image
inference_image = sagemaker.image_uris.retrieve(
    framework="huggingface",
    region=REGION,
    version="4.37.0",
    py_version="py310",
    image_scope="inference",
    instance_type="ml.m5.xlarge",
    base_framework_version="pytorch2.1.0",  # Required for HuggingFace images
)

step_register = RegisterModel(
    name="RegisterModel",
    estimator=huggingface_estimator,
    model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    content_types=["application/json"],
    response_types=["application/json"],
    inference_instances=["ml.m5.large", "ml.m5.xlarge", "ml.g4dn.xlarge"],
    transform_instances=["ml.m5.xlarge"],
    model_package_group_name="sg-finetune-models",
    approval_status=approval_status,
    image_uri=inference_image,
)

print("Step 3: RegisterModel configured")
print(f"Inference image: {inference_image}")

## Create and Execute Pipeline

In [ ]:
# Create pipeline
pipeline = Pipeline(
    name=PIPELINE_NAME,
    parameters=[
        num_examples,
        train_ratio,
        model_id,
        instance_type,
        epochs,
        batch_size,
        learning_rate,
        max_length,
        approval_status,
    ],
    steps=[step_preprocess, step_train, step_register],
    sagemaker_session=sagemaker_session,
)

print(f"Pipeline '{PIPELINE_NAME}' created with 3 steps")

In [ ]:
# View pipeline definition (optional)
import json
pipeline_def = json.loads(pipeline.definition())
print(json.dumps(pipeline_def, indent=2)[:2000] + "...")

In [ ]:
# Create/update pipeline in SageMaker
response = pipeline.upsert(role_arn=role)
print(f"Pipeline ARN: {response['PipelineArn']}")

In [ ]:
# Execute pipeline with default parameters
execution = pipeline.start()
print(f"Execution ARN: {execution.arn}")
print(f"\nMonitor at: https://{REGION}.console.aws.amazon.com/sagemaker/home?region={REGION}#/pipelines/{PIPELINE_NAME}/executions")

In [ ]:
# Optional: Execute with custom parameters
# execution = pipeline.start(
#     parameters={
#         "NumExamples": 200,
#         "Epochs": 3,
#         "BatchSize": 4,
#     }
# )
# print(f"Execution ARN: {execution.arn}")

In [ ]:
# Wait for execution to complete (optional)
# execution.wait()

In [ ]:
# Check execution status
execution.describe()

## Cleanup (Optional)

In [ ]:
# Delete pipeline (optional)
# sm_client = boto3.client("sagemaker", region_name=REGION)
# sm_client.delete_pipeline(PipelineName=PIPELINE_NAME)
# print(f"Pipeline '{PIPELINE_NAME}' deleted")